# Many recordings

A study is a list of recordings, and scoring it is the same few decisions
repeated. There are two ways through that list, and they are for different jobs.

**Interactive** — for scoring a study for the first time. Every recording needs
looking at, so the queue hands them out one at a time and rewrites `config.json`
as it goes. Your notebook reads that one file and never changes.

**Headless** — for reproducing a result. No interaction, everything as written.
Point it at saved `run.json` files and every decision is already recorded.

Both remember where you got to, so an interrupted session resumes.

**Runs as-is, with no data** — section 2 builds a small synthetic study.

## 1. Setup

In [ ]:
import os

for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import json
import shutil

import numpy as np
import pandas as pd

import nyx

## 2. The manifest

A batch is driven by a CSV, one row per recording. Labs already keep the
recording list in a spreadsheet, so this is usually a matter of renaming a few
columns.

Only `recording_path` is required. Everything else falls back to the defaults you
pass to `open_queue`:

| column | meaning |
|---|---|
| `recording_path` | the signal file |
| `eeg_channel`, `emg_channel` | index or channel name |
| `annotation_path` | reference scoring; blank means none |
| `params` | parameter file for this recording |
| `window_start`, `window_end` | analysis window, seconds |
| `name` | output folder name |
| `dataset`, `recording_name`, `data_root` | a dataset recipe, instead of paths |

Below we synthesise three recordings and write a manifest for them. With real
data you would write this CSV once, by hand or from your lab's database.

In [ ]:
STUDY = "demo_output/03_batch_queue"
shutil.rmtree(STUDY, ignore_errors=True)
os.makedirs(f"{STUDY}/data", exist_ok=True)

# Three "animals", each a different draw of the synthetic signal.
rows = []
for i in range(3):
    eeg, emg, fs, hypnogram = nyx.demo.make_demo_signals(seed=i)
    path = f"{STUDY}/data/m{i:02d}.npz"
    np.savez(path, eeg=eeg, emg=emg, fs=fs)

    # A reference scoring, so the batch has something to report agreement against.
    scores = f"{STUDY}/data/m{i:02d}_scores.csv"
    pd.DataFrame(hypnogram).to_csv(scores, index=False)

    rows.append({
        "recording_path": os.path.abspath(path),
        "annotation_path": os.path.abspath(scores),
        "annotation_format": "interval_csv",
        "name": f"m{i:02d}",
    })

MANIFEST = f"{STUDY}/recordings.csv"
nyx.write_manifest(MANIFEST, rows)

print(open(MANIFEST).read())

The demo params are not a file on disk, so write them out — a real run would
point at `../params/mouse.json`.

In [ ]:
PARAMS = f"{STUDY}/params.json"
with open(PARAMS, "w") as handle:
    json.dump(nyx.demo_params(), handle, indent=4)

## 3. Open it as a queue

In [ ]:
queue = nyx.open_queue(
    MANIFEST,
    params=PARAMS,
    output_root=f"{STUDY}/results",
    config_path=f"{STUDY}/config.json",
)
queue.print_status()

## 4. Interactive — one recording at a time

`queue.next()` hands out the next unscored recording **and writes
`config.json`** pointing at it. That is the part that matters: your scoring
notebook reads that one file, so you never edit a path between recordings.

In practice you would now switch to `01_score_recording.ipynb`, run it top to
bottom, adjust what needs adjusting, and come back. Here we do the equivalent
inline.

In [ ]:
config = queue.next()
print("next up:", config.name)
print("wrote:  ", queue.config_path)

This is what the scoring notebook does — the same two lines every time,
whichever recording is current:

In [ ]:
config = nyx.load_config(f"{STUDY}/config.json")
recording, reference = config.load()
print(recording.describe())

In [ ]:
result = nyx.score_recording(recording, config.params, window=config.window,
                             reference=reference, verbose=False)
nyx.save_results(result, config.output_dir, config=config, plots=False)

print(f"MF1 {result.agreement.mf1:.3f}")

Then mark it and move on. `done()` when you are happy with the result;
`skip(reason)` when you are not going to score it at all.

**`skip` requires a reason and keeps it.** That is the difference between a
deliberate omission and a gap nobody can account for later — "why is there no
result for m01" should have an answer next year.

In [ ]:
queue.done()

config = queue.next()
print("next up:", config.name)

# Pretend this one turned out to be unusable.
queue.skip("EMG disconnected halfway through")

queue.print_status()

### Or as a loop

Same thing, without calling `next()` yourself. The loop refuses to hand out a
recording you left unmarked, rather than cycling on it forever:

```python
for config in queue:
    recording, reference = config.load()
    result = nyx.score_recording(recording, config.params, reference=reference)
    nyx.save_results(result, config.output_dir, config=config)
    queue.done()
```

## 5. Resuming

Progress lives in a small JSON file beside the results. Reopening the same
manifest picks up where you left off — which matters, because scoring a study
takes days and kernels do not.

In [ ]:
reopened = nyx.open_queue(
    MANIFEST,
    params=PARAMS,
    output_root=f"{STUDY}/results",
    config_path=f"{STUDY}/config.json",
)
reopened.print_status()

It is also self-healing. A recording whose output folder holds a `run.json`
counts as scored **whatever the state file says** — so the queue is still right
after you delete that file, or when a colleague scored part of the list on
another machine and you copied the results across.

## 6. Headless — reproducing a result

`run_all()` scores everything remaining with no interaction.

This is how you **reproduce**: point the manifest at saved `run.json` files and
every decision is already in them, so nothing is left to choose. Running it on
*unseen* recordings scores them on defaults, which are a starting point rather
than an answer — that is what section 4 is for.

In [ ]:
summary = reopened.run_all(verbose=True)

print()
for row in summary:
    print(row)

A recording that fails is recorded as `failed` with its error, and the batch
carries on — one unreadable file should not cost you the other forty. Pass
`stop_on_error=True` when you want the traceback for one you are fixing.

In [ ]:
reopened.print_status()

## 7. What you end up with

One folder per recording, each holding the hypnogram, the figures and a
`run.json` that replays the scoring exactly.

In [ ]:
results = f"{STUDY}/results"
for name in sorted(os.listdir(results)):
    path = os.path.join(results, name)
    if os.path.isdir(path):
        print(f"{name}/")
        for item in sorted(os.listdir(path)):
            print(f"    {item}")

### Collecting the results

Agreement metrics across the study, for a figure or a table:

```python
import pandas as pd

rows = []
for config in queue.scored:
    record = nyx.load_json(os.path.join(config.output_dir, "run.json"))
    rows.append({"recording": config.name, **record.get("decisions", {}).get("emg", {})})

pd.DataFrame(rows)
```

`run_all` also returns one row per recording with its MF1, which is usually
enough:

```python
pd.DataFrame(queue.run_all())
```

---

Next: `04_human_five_stage.ipynb` for multi-step scoring, or `params/README.md`
for what to put in a parameter file.